In [ ]:
# Inputs: DataFrame of wildfires

# Outputs: Wild_Occurrence_Line_detail 

# Purpose: Create the fire sequences between each fire occurrence point and each urban area

# Notable functions:# find all fires within 10km of fire occurrence; calculate distance and direction; 
                    # draw line between them; break up line into points; keep only wildfires 
                    # get occurrence_burn_area output for GEE

In [ ]:
import pandas as pd 
pd.set_option('display.max_columns', None)
import pyproj
pyproj.network.set_network_enabled(False)
from pyproj import Proj, Transformer
from pyproj import Geod
import geopandas

In [ ]:
def convert_to_epsg(df, df2, epsg_int):
    df = df.to_crs(f"EPSG:{epsg_int}")
    df2.to_crs(df.crs, inplace=True)
    return df, df2

def create_points(row, geometry, point_separation):
    import shapely
    import numpy as np
    geom = row[geometry]
    if geom is None or geom.is_empty:
        return []
    point_list = [geom.interpolate(distance=x) for x in np.arange(start=0, stop=geom.length, step=point_separation)]
    return point_list
    
    

In [ ]:
def find_nearby_fires(fires_dataframe, urban_file, buffer_size):
    u_df = geopandas.read_file(urban_file)
    f_df, u_df = convert_to_epsg(fires_dataframe, u_df, 5070)
    
    u_short = u_df[['geometry']]
    u_short["UrbanGeom"] = u_short["geometry"]
    
    f_df = f_df[f_df["FIRE_TYPE"] == 'Wildfire']
    f_df["fires_df_buffer"] = f_df.buffer(buffer_size)
    
    f_short = f_df[["FIRE_ID", "fires_df_buffer"]]
    f_short.rename(columns={"fires_df_buffer": "geometry"}, inplace=True)
    
    fires_2 = geopandas.sjoin(f_short, u_short, how='left')
    
    f_urb = pd.merge(f_df, fires_2, on="FIRE_ID", how='left')
    
    f_urb = f_urb.reset_index(drop=True)
    f_urb["Total_Distance_To_Urban"] = f_urb["geometry_x"].distance(f_urb["UrbanGeom"])
    
    f_urb = f_urb[f_urb['Total_Distance_To_Urban'].notna()]
    
    return f_urb
   
    
    

In [ ]:
occurrences_df = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartOneOutput.shp")


occurrences_df.head()

In [ ]:
occ_2020 = occurrences_df[occurrences_df["IG_DATE"] > "2019-12-31"]
occ_2010 = occurrences_df[(occurrences_df["IG_DATE"] > "2009-12-31") & (occurrences_df["IG_DATE"] <= "2019-12-31")]
occ_2000 = occurrences_df[occurrences_df["IG_DATE"] <= "2009-12-31"]


In [ ]:
urban_2000 = r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\WUI\Urban Areas\tl_2008_us_uac00.shp"
urban_2010 = r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\WUI\Urban Areas\tl_2010_us_uac10.shp"
urban_2020 = r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\WUI\Urban Areas\tl_2020_us_uac20.shp"

In [ ]:
occ_2000 = find_nearby_fires(occ_2000, urban_2000, 10000)
occ_2010 = find_nearby_fires(occ_2010, urban_2010, 10000)
occ_2020 = find_nearby_fires(occ_2020, urban_2020, 10000)

In [ ]:
fires_df_urb = pd.concat([occ_2000, occ_2010, occ_2020])

In [ ]:
fires_df_urb.head()

In [ ]:
len(fires_df_urb)

In [ ]:
transformer = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)
lon1, lat1 = transformer.transform(fires_df_urb["geometry_x"].x, fires_df_urb["geometry_x"].y)
centroids = fires_df_urb["UrbanGeom"].centroid
lon2, lat2 = transformer.transform(centroids.x, centroids.y)

geod = Geod(ellps="WGS84")
from_angle, to_angle, distance = geod.inv(lon1, lat1, lon2, lat2)
fires_df_urb["Direction_To_Urban"] = from_angle
fires_df_urb["Direction_From_Urban_To_Fire"] = to_angle
fires_df_urb["Distance_To_Urban"] = distance

angles = [i for i in range(-1, 360, 20)]

fires_df_urb["Angles"] = fires_df_urb["Direction_To_Urban"].astype(float).apply(lambda x: [((x + a + 180) % 360) - 180 for a in angles])

fires_df_urb.head()


In [ ]:
fires_df_urb = fires_df_urb.explode("Angles")
fires_df_urb["Angles"] = fires_df_urb["Angles"].astype(float)
fires_df_urb["Dummy_Distance"] = 10000
fires_df_urb.head()

In [ ]:
lon3, lat3 = transformer.transform(fires_df_urb["geometry_x"].x, fires_df_urb["geometry_x"].y)
angles_column = fires_df_urb["Angles"]
distance_column = fires_df_urb["Dummy_Distance"]
print(len(lon3), len(lat3), len(angles_column))

In [ ]:
lon4, lat4, _ = geod.fwd(lon3, lat3, angles_column, distance_column)
fires_df_urb["ConstructPoint_Lon"] = lon4
fires_df_urb["ConstructPoint_Lat"] = lat4

fires_df_urb.head()

In [ ]:
fires_df_urb["TestPoint"] = geopandas.points_from_xy(fires_df_urb["ConstructPoint_Lon"].astype(float),fires_df_urb["ConstructPoint_Lat"].astype(float), crs=5070)
fires_df_urb = geopandas.GeoDataFrame(fires_df_urb, geometry="TestPoint", crs=4326)
fires_df_urb["TestPoint"] = fires_df_urb["TestPoint"].to_crs(5070)
fires_df_urb.head()

In [ ]:
fires_df_urb["IgnitionUrbanLine"] = fires_df_urb['geometry_x'].shortest_line(fires_df_urb["TestPoint"])
fires_df_urb.head()

In [ ]:
fires_df_urb = fires_df_urb[fires_df_urb["UrbanGeom"].intersects(fires_df_urb["IgnitionUrbanLine"])]
fires_df_urb.drop_duplicates(subset=["FIRE_ID", "ConstructPoint_Lon", "ConstructPoint_Lat"], inplace=True)
fires_df_urb.head()

In [ ]:
fires_df_urb = fires_df_urb[['FIRE_ID', 'FIRE_TYPE', 'IG_DATE', 'Total_Distance_To_Urban', 'Distance_To_Urban', 'Angles', 'geometry_x', 'UrbanGeom', 'TestPoint', 'IgnitionUrbanLine', 'CrossedWUI']].reset_index()
fires_df_urb.rename(columns={"geometry_x": "geometry", "TestPoint": "LineEndPoint"}, inplace=True)
fires_df_urb.head(100)

In [ ]:
burn_df = geopandas.read_file(r"C:\Users\jezkn\Local\Data Science Projects\MSc Project Data\Data\Burn Area Boundary\S_USA.MTBS_BURN_AREA_BOUNDARY.shp")
burn_df.crs

In [ ]:
#fires_df_urb.set_geometry('')
burn_df = burn_df.to_crs(5070)

In [ ]:
fires_df_urb = pd.merge(fires_df_urb, burn_df[['FIRE_ID', 'geometry']], on='FIRE_ID')
fires_df_urb.rename(columns={'geometry_y': 'Burn_Area'}, inplace=True)
fires_df_urb.head()

In [ ]:
fires_df_urb["OccurrenceUrbanID"] = fires_df_urb.index + 1
fires_df_urb.head(100)

In [ ]:
len(fires_df_urb)

In [ ]:
fires_df_urb["point_list"] = fires_df_urb.apply(lambda x: create_points(row=x, geometry="IgnitionUrbanLine", point_separation=50), axis=1)
fires_df_urb.head()

In [ ]:
df_points = fires_df_urb.explode(column="point_list")
df_points['point_order'] = df_points.groupby(['FIRE_ID', 'OccurrenceUrbanID']).cumcount()
df_points.head()

In [ ]:
df_points.groupby(["OccurrenceUrbanID"]).count()

In [ ]:
df_points

In [ ]:
df_points["Y"] = df_points.apply(lambda x: 1 if pd.notnull(x['point_list']) and x['point_list'].intersects(x["Burn_Area"]) else 0, axis=1)

In [ ]:
df_points["IsUrban"] = df_points.apply(lambda x: 1 if pd.notnull(x['point_list']) and x['point_list'].intersects(x["UrbanGeom"]) else 0, axis=1)

In [ ]:
df_points["WUIBreach"] = df_points.apply(lambda x: 1 if x["Y"] == 1 and x["IsUrban"] == 1 else 0, axis=1)


In [ ]:
df_points["Y"].value_counts()

In [ ]:
df_points["IsUrban"].value_counts()

In [ ]:
df_points["WUIBreach"].value_counts()

In [ ]:
last_urban_point = (df_points[df_points["IsUrban"] == 1].groupby("OccurrenceUrbanID")["point_order"].max())

In [ ]:
df_points["last_urban_point"] = df_points["OccurrenceUrbanID"].map(last_urban_point)

In [ ]:
df_points["RemoveFlag"] = (df_points["point_order"] > df_points["last_urban_point"]).astype(int)

In [ ]:
df_points = df_points[df_points["RemoveFlag"] == 0]

In [ ]:
df_points["Y"].value_counts()

In [ ]:
df_points.head(1000)

In [ ]:
df_out = df_points.copy()

df_out['geometry'] = df_out['point_list']
df_out = geopandas.GeoDataFrame(df_out, geometry=df_out['point_list'], crs="EPSG:5070")

df_out = df_out[["FIRE_ID", "IG_DATE", "FIRE_TYPE", "OccurrenceUrbanID", "point_order", "geometry"]]

In [ ]:
df_out

In [ ]:
df_points.rename(columns={"point_list": "geometry", "Angles": "UrbanAngle", "NAME20": "City"}, inplace=True)
df_points.drop(columns=['geometry_x', 'UrbanGeom', 'LineEndPoint', 'Total_Distance_To_Urban', 'Distance_To_Urban', 'IgnitionUrbanLine', 'CrossedWUI', 'Burn_Area', 'RemoveFlag'], inplace=True)
df_points.head()

In [ ]:
df_points.drop(columns=["index"], inplace=True)
df_points = geopandas.GeoDataFrame(df_points, geometry=df_points['geometry'], crs="EPSG:5070")

In [ ]:
df_out.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartTwoOutput3.shp")

In [ ]:
df_points.to_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\FINAL_PartTwoOutputShape3.shp")